# Reading & Writing Data

Data scientists work with data from many sources. This notebook covers reading from and writing to the most common formats:

- CSV (most common)
- JSON
- Excel
- SQLite / SQL databases
- Parquet (columnar format)
- Web (HTML tables, APIs)

In [1]:
import pandas as pd
import numpy as np
import json
import os

# Helper — create a sample DataFrame we'll save and reload in multiple formats
np.random.seed(0)
df_orig = pd.DataFrame({
    'id':       range(1, 11),
    'name':     ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank','Ivy','Jack'],
    'dept':     ['Eng','Mkt','Eng','HR','Mkt','Eng','HR','Mkt','Eng','HR'],
    'salary':   np.random.randint(50000, 120000, 10),
    'joined':   pd.date_range('2020-01-01', periods=10, freq='3ME'),
    'active':   [True]*7 + [False]*3
})
print(df_orig)

   id   name dept  salary     joined  active
0   1  Alice  Eng  118268 2020-01-31    True
1   2    Bob  Mkt   93567 2020-04-30    True
2   3  Carol  Eng   92613 2020-07-31    True
3   4   Dave   HR   95891 2020-10-31    True
4   5    Eve  Mkt   71243 2021-01-31    True
5   6  Frank  Eng   91993 2021-04-30    True
6   7  Grace   HR  105026 2021-07-31    True
7   8   Hank  Mkt   98600 2021-10-31   False
8   9    Ivy  Eng   89512 2022-01-31   False
9  10   Jack   HR  102620 2022-04-30   False


## 1. CSV — Comma Separated Values

In [2]:
# Write CSV
df_orig.to_csv('/tmp/employees.csv', index=False)
print('Saved CSV')

# Read CSV — most common params
df_csv = pd.read_csv('/tmp/employees.csv')
print(df_csv.dtypes)
print(df_csv.head(3))

Saved CSV
id        int64
name        str
dept        str
salary    int64
joined      str
active     bool
dtype: object
   id   name dept  salary      joined  active
0   1  Alice  Eng  118268  2020-01-31    True
1   2    Bob  Mkt   93567  2020-04-30    True
2   3  Carol  Eng   92613  2020-07-31    True


In [3]:
# Useful read_csv parameters
pd.read_csv('/tmp/employees.csv',
    usecols=['name','dept','salary','joined'],   # only load these columns
    dtype={'salary': np.float32},       # override dtype
    parse_dates=['joined'],             # parse as datetime
    nrows=5,                            # read only first 5 rows
)

# Reading in chunks (for large files)
total_salary = 0
for chunk in pd.read_csv('/tmp/employees.csv', chunksize=3):
    total_salary += chunk['salary'].sum()
print('Total salary (chunked):', total_salary)

# Other delimiters
df_orig.to_csv('/tmp/employees.tsv', sep='\t', index=False)
df_tsv = pd.read_csv('/tmp/employees.tsv', sep='\t')
print(df_tsv.shape)

Total salary (chunked): 959333
(10, 6)


## 1b. From-Scratch — Minimal CSV Parser

`pd.read_csv` looks like "just split every line on a comma," but real CSVs need quoting-aware
parsing (a comma *inside* a quoted field is not a delimiter). A minimal hand-rolled parser
below shows exactly what breaks with the naive approach, and what a quoting-aware version has
to track, before comparing both to `pd.read_csv`'s output on the same text.

In [4]:
import io

def naive_split_parse(csv_text):
    """Splits every line on ',' -- breaks on a quoted field containing a comma."""
    return [line.split(',') for line in csv_text.strip().split('\n')]

def quote_aware_parse(csv_text):
    """A minimal quote-aware CSV parser: honors "..." fields that may contain commas."""
    rows = []
    for line in csv_text.strip().split('\n'):
        fields, field, in_quotes = [], [], False
        for ch in line:
            if ch == '"':
                in_quotes = not in_quotes
            elif ch == ',' and not in_quotes:
                fields.append(''.join(field))
                field = []
            else:
                field.append(ch)
        fields.append(''.join(field))
        fields = [f[1:-1] if f.startswith('"') and f.endswith('"') else f for f in fields]
        rows.append(fields)
    return rows

sample_csv = 'name,city,note\nAlice,NYC,"likes coffee, tea"\nBob,LA,"no comment"'

print('Naive split (breaks on the embedded comma -- 4 fields instead of 3):')
for row in naive_split_parse(sample_csv):
    print(row)

print('\nQuote-aware parse (correct -- 3 fields per row):')
for row in quote_aware_parse(sample_csv):
    print(row)

print('\npd.read_csv (handles this automatically):')
print(pd.read_csv(io.StringIO(sample_csv)))

Naive split (breaks on the embedded comma -- 4 fields instead of 3):
['name', 'city', 'note']
['Alice', 'NYC', '"likes coffee', ' tea"']
['Bob', 'LA', '"no comment"']

Quote-aware parse (correct -- 3 fields per row):
['name', 'city', 'note']
['Alice', 'NYC', 'likes coffee, tea']
['Bob', 'LA', 'no comment']

pd.read_csv (handles this automatically):
    name city               note
0  Alice  NYC  likes coffee, tea
1    Bob   LA         no comment


## 2. JSON

In [5]:
# Write JSON
df_orig.to_json('/tmp/employees.json', orient='records', indent=2, date_format='iso')

# Read JSON
df_json = pd.read_json('/tmp/employees.json')
print(df_json.dtypes)
print(df_json.head(3))

# Nested JSON (API-style response)
import json

api_response = {
    'status': 'ok',
    'count': 3,
    'data': [
        {'id': 1, 'name': 'Alice', 'tags': ['python', 'ml']},
        {'id': 2, 'name': 'Bob',   'tags': ['sql', 'bi']},
        {'id': 3, 'name': 'Carol', 'tags': ['python', 'stats']}
    ]
}

# Normalise nested JSON into a flat table
df_api = pd.json_normalize(api_response['data'])
print(df_api)

id        int64
name        str
dept        str
salary    int64
joined      str
active     bool
dtype: object
   id   name dept  salary                   joined  active
0   1  Alice  Eng  118268  2020-01-31T00:00:00.000    True
1   2    Bob  Mkt   93567  2020-04-30T00:00:00.000    True
2   3  Carol  Eng   92613  2020-07-31T00:00:00.000    True
   id   name             tags
0   1  Alice     [python, ml]
1   2    Bob        [sql, bi]
2   3  Carol  [python, stats]


## 3. Excel

In [6]:
# Write Excel (requires openpyxl: uv add openpyxl)
try:
    df_orig.to_excel('/tmp/employees.xlsx', sheet_name='Employees', index=False)

    # Read Excel
    df_excel = pd.read_excel('/tmp/employees.xlsx', sheet_name='Employees')
    print(df_excel.head(3))

    # Write multiple sheets
    with pd.ExcelWriter('/tmp/multi_sheet.xlsx', engine='openpyxl') as writer:
        df_orig[df_orig['dept']=='Eng'].to_excel(writer, sheet_name='Engineering', index=False)
        df_orig[df_orig['dept']=='HR'].to_excel(writer,  sheet_name='HR', index=False)

    print('Multi-sheet Excel written')
except ImportError:
    print('Install openpyxl: uv add openpyxl')

   id   name dept  salary     joined  active
0   1  Alice  Eng  118268 2020-01-31    True
1   2    Bob  Mkt   93567 2020-04-30    True
2   3  Carol  Eng   92613 2020-07-31    True
Multi-sheet Excel written


## 4. SQLite

In [7]:
import sqlite3

# Write DataFrame directly to SQLite
conn = sqlite3.connect('/tmp/company.db')
df_orig.to_sql('employees', conn, if_exists='replace', index=False)
print('Written to SQLite')

# Read with SQL query
df_sql = pd.read_sql('SELECT * FROM employees WHERE dept = "Eng"', conn)
print(df_sql)
conn.close()

Written to SQLite
   id   name dept  salary               joined  active
0   1  Alice  Eng  118268  2020-01-31 00:00:00       1
1   3  Carol  Eng   92613  2020-07-31 00:00:00       1
2   6  Frank  Eng   91993  2021-04-30 00:00:00       1
3   9    Ivy  Eng   89512  2022-01-31 00:00:00       0


In [8]:
# More complex queries
conn = sqlite3.connect('/tmp/company.db')

df_agg = pd.read_sql('''
    SELECT dept,
           COUNT(*)         AS headcount,
           AVG(salary)      AS avg_salary,
           MAX(salary)      AS max_salary
    FROM   employees
    GROUP  BY dept
    ORDER  BY avg_salary DESC
''', conn)
print(df_agg)
conn.close()

  dept  headcount     avg_salary  max_salary
0   HR          3  101179.000000      105026
1  Eng          4   98096.500000      118268
2  Mkt          3   87803.333333       98600


## 5. Parquet — High-Performance Columnar Format

Parquet is the preferred format for large datasets — much faster to read/write than CSV, preserves dtypes.

In [9]:
# Requires pyarrow: uv add pyarrow
try:
    df_orig.to_parquet('/tmp/employees.parquet', index=False)
    df_parquet = pd.read_parquet('/tmp/employees.parquet')
    print(df_parquet.dtypes)  # dtypes are preserved!
    print(df_parquet.head(3))
except ImportError:
    print('Install pyarrow: uv add pyarrow')

id                 int64
name                 str
dept                 str
salary             int64
joined    datetime64[us]
active              bool
dtype: object
   id   name dept  salary     joined  active
0   1  Alice  Eng  118268 2020-01-31    True
1   2    Bob  Mkt   93567 2020-04-30    True
2   3  Carol  Eng   92613 2020-07-31    True


## 6. Reading HTML Tables from the Web

In [10]:
# pd.read_html() extracts all <table> elements from an HTML page or string
# (pandas 2.1+ requires wrapping a literal HTML string in io.StringIO -- passing the
# raw string directly is treated as a file path/URL and raises FileNotFoundError)
sample_html = '''
<table>
  <thead><tr><th>Country</th><th>GDP_bn</th><th>Population_m</th></tr></thead>
  <tbody>
    <tr><td>USA</td><td>25000</td><td>335</td></tr>
    <tr><td>China</td><td>18000</td><td>1412</td></tr>
    <tr><td>Germany</td><td>4200</td><td>84</td></tr>
  </tbody>
</table>
'''

tables = pd.read_html(io.StringIO(sample_html))
df_html = tables[0]
print(df_html)

# To read from a real URL (needs internet):
# tables = pd.read_html('https://en.wikipedia.org/wiki/List_of_countries_by_GDP')
# print(len(tables), 'tables found')

   Country  GDP_bn  Population_m
0      USA   25000           335
1    China   18000          1412
2  Germany    4200            84


## 7. Saving Data — Best Practices

| Format | Use when | Pros | Cons |
|--------|----------|------|------|
| CSV | Sharing, interop | Universal | Slow, no dtype preservation |
| JSON | APIs, nested data | Human-readable | Slow for large data |
| Excel | Non-technical users | Familiar | Slow, large files |
| Parquet | Big data, pipelines | Fast, small, preserves dtypes | Not human-readable |
| SQLite | Relational queries | SQL support | Single-file DB only |

In [11]:
# Performance comparison: CSV vs Parquet
import time

# Create a larger DataFrame
big_df = pd.DataFrame(np.random.randn(100_000, 10), columns=list('ABCDEFGHIJ'))

# CSV write/read time
start = time.perf_counter()
big_df.to_csv('/tmp/big.csv', index=False)
t_csv_write = time.perf_counter() - start

start = time.perf_counter()
pd.read_csv('/tmp/big.csv')
t_csv_read = time.perf_counter() - start

# Parquet write/read time
try:
    start = time.perf_counter()
    big_df.to_parquet('/tmp/big.parquet', index=False)
    t_parq_write = time.perf_counter() - start

    start = time.perf_counter()
    pd.read_parquet('/tmp/big.parquet')
    t_parq_read = time.perf_counter() - start

    print(f'CSV   write: {t_csv_write:.3f}s  |  read: {t_csv_read:.3f}s')
    print(f'Parquet write: {t_parq_write:.3f}s  |  read: {t_parq_read:.3f}s')
except ImportError:
    print(f'CSV write: {t_csv_write:.3f}s  read: {t_csv_read:.3f}s')
    print('Install pyarrow for Parquet comparison')

CSV   write: 0.693s  |  read: 0.102s
Parquet write: 0.062s  |  read: 0.014s


## 8. Failure Modes

- **Encoding mismatch (UTF-8 vs Latin-1).** A file written in one encoding, read assuming
  another, either raises `UnicodeDecodeError` or silently produces mojibake.
- **Silent type misinference on a numeric-looking ID column.** An ID column that happens to
  look numeric (`00123`) is read as `int64` by default — the leading zeros are gone, silently,
  with no error.

In [12]:
text = 'Héllo Wörld - café'
with open('/tmp/latin1_demo.csv', 'w', encoding='latin-1') as f:
    f.write('name\n' + text + '\n')

try:
    df_wrong = pd.read_csv('/tmp/latin1_demo.csv', encoding='utf-8')
    print('utf-8 read (wrong encoding):')
    print(df_wrong)
except UnicodeDecodeError as e:
    print('UnicodeDecodeError reading with utf-8:', e)

df_right = pd.read_csv('/tmp/latin1_demo.csv', encoding='latin-1')
print('\nlatin-1 read (correct encoding):')
print(df_right)

UnicodeDecodeError reading with utf-8: 'utf-8' codec can't decode byte 0xe9 in position 1: invalid continuation byte

latin-1 read (correct encoding):
                 name
0  Héllo Wörld - café


In [13]:
ids_csv = 'employee_id,name\n00123,Alice\n00456,Bob\n00789,Carol'

df_ids = pd.read_csv(io.StringIO(ids_csv))
print('Default read:')
print(df_ids)
print(df_ids.dtypes)
print('Leading zeros lost:', df_ids['employee_id'].tolist())

df_ids_str = pd.read_csv(io.StringIO(ids_csv), dtype={'employee_id': str})
print('\nWith dtype={"employee_id": str}:')
print(df_ids_str)
print(df_ids_str['employee_id'].tolist())

Default read:
   employee_id   name
0          123  Alice
1          456    Bob
2          789  Carol
employee_id    int64
name             str
dtype: object
Leading zeros lost: [123, 456, 789]

With dtype={"employee_id": str}:
  employee_id   name
0       00123  Alice
1       00456    Bob
2       00789  Carol
['00123', '00456', '00789']


## Quick Summary

| Format | Read | Write |
|--------|------|-------|
| CSV | `pd.read_csv('file.csv')` | `df.to_csv('file.csv', index=False)` |
| JSON | `pd.read_json('file.json')` | `df.to_json('file.json', orient='records')` |
| Excel | `pd.read_excel('file.xlsx')` | `df.to_excel('file.xlsx', index=False)` |
| SQLite | `pd.read_sql('SELECT ...', conn)` | `df.to_sql('table', conn)` |
| Parquet | `pd.read_parquet('file.parquet')` | `df.to_parquet('file.parquet')` |
| HTML | `pd.read_html(url)` | — |

**Next →** [05 – Matplotlib](../05-matplotlib/)